%md
###### 17_nlp_model_training

###### Purpose
Train multiple Support Ticket NLP classification models using a TF-IDF + Scikit-learn pipeline, log experiments with MLflow, and store model evaluation results for model selection in a Unity Catalog Delta table.

###### Input
Inputs
- Ticket templates used to generate a balanced dataset
- Bronze, Silver and Gold Delta tables
- Cleaned Gold table converted to Pandas
- Train/Test split
- TF-IDF Vectorizer
- Scikit-learn Pipeline

######  Output

- MLflow experiment containing:
    - Parameters
    - Metrics
    - Logged model
    - Signature
    - Input Example

- Delta table:
    - dbw_agentic_ai_dev.support_ticket_ai.model_training_results


- Models Trained:
    - Logistic Regression
    - Decision Tree
    - Random Forest

######  Architecture

```text

Raw Ticket Templates
      ↓
Bronze Delta Table
      ↓
Silver Delta Table
      ↓
Gold Delta Table
      ↓
Prepare Training Data
      ↓
TF-IDF + Pipeline
      ↓
Train Models
      ↓
Evaluate Models
      ↓
MLflow Tracking
   • Parameters
   • Metrics
   • Signature
   • Input Example
   • Logged Model
      ↓
Results Delta Table


```

###### Skills Covered

Delta Lake, TF-IDF, Scikit-learn Pipeline, Logistic Regression, Decision Tree, Random Forest, MLflow, Model Signature, Model Logging and Unity Catalog.

###### Section 1 : Create Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dbw_agentic_ai_dev.support_ticket_ai;

###### Section 2 : Import Libraries

In [0]:
import random
from pyspark.sql import Row
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
import pandas as pd
import mlflow
import joblib
import tempfile
import mlflow.pyfunc

###### Section 3 : Generate Sample Dataset

In [0]:
#Use templates to generate a larger balanced dataset.

random.seed(42)

ticket_templates = {
    "Technical": [
        "Internet is slow",
        "Wifi keeps disconnecting",
        "Connection drops every evening",
        "Router is not working",
        "Internet speed is very poor",
        "Service outage in my area",
        "Modem keeps restarting",
        "Video streaming keeps buffering",
        "Network connection is unstable",
        "Internet disconnects during meetings"
    ],
    "Login": [
        "Unable to login to my account",
        "Password reset is not working",
        "Account locked after many attempts",
        "Cannot access my customer portal",
        "Login page keeps failing",
        "Forgot my password",
        "Two factor authentication is not working",
        "My account access is blocked",
        "Unable to sign in",
        "Login error on mobile app"
    ],
    "Billing": [
        "My bill is too high",
        "I was charged incorrectly",
        "Unexpected charges on my invoice",
        "Refund not received",
        "Payment was processed twice",
        "Monthly bill amount is wrong",
        "Need explanation for extra fees",
        "Billing statement has incorrect amount",
        "Autopay charged the wrong card",
        "Invoice shows duplicate charge"
    ],
    "Cancellation": [
        "I want to cancel my service",
        "Please terminate my account",
        "I no longer need this service",
        "Cancel my subscription",
        "Close my account immediately",
        "I want to switch providers",
        "Stop my service next month",
        "I am leaving this provider",
        "Please disconnect my service",
        "I want to end my contract"
    ]
}


###### Section 4 : Create Bronze Table

In [0]:
support_data = []
ticket_id = 1

for category, templates in ticket_templates.items():
    for i in range(25):
        text = random.choice(templates)
        support_data.append(
            Row(
                ticket_id=f"T{ticket_id:03d}",
                ticket_text=text,
                category=category
            )
        )
        ticket_id += 1

bronze_df = spark.createDataFrame(support_data)

bronze_df.write.mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.bronze_support_tickets")

###### Section 5 : Create Silver Table

In [0]:
silver_df = spark.sql("""
SELECT
    ticket_id,
    LOWER(TRIM(ticket_text)) AS cleaned_ticket_text,
    category
FROM dbw_agentic_ai_dev.support_ticket_ai.bronze_support_tickets
WHERE ticket_text IS NOT NULL
  AND category IS NOT NULL
""")

silver_df.write.mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.silver_support_tickets")

###### Section 6 : Create Gold Table

In [0]:
gold_df = spark.sql("""
SELECT
    ticket_id,
    cleaned_ticket_text,
    category
FROM dbw_agentic_ai_dev.support_ticket_ai.silver_support_tickets
""")

gold_df.write.mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_features")

# Read Gold table into pandas
df = spark.table( "dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_features").toPandas()

###### Section 7 : Prepare Training Data

In [0]:
#Train/test split
X = df["cleaned_ticket_text"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

###### Section 8 : Train Models, Log MLflow Artifacts and Save Results

In [0]:
# Step 9: TF-IDF +  pipeline

user_name = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

#Build experiment path
experiment_path = f"/Users/{user_name}/support_ticket_experiment"
mlflow.set_experiment(experiment_path)

#print(experiment_path)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

for model_name, classifier in models.items():

    #MLflow = Complete Experiment History.This stores everything about the experiment.
    #Start MLflow run
    with mlflow.start_run(run_name=model_name):
         
        pipeline = Pipeline(
            steps=[
                ("tfidf", TfidfVectorizer(stop_words="english")),
                ("classifier", classifier)
            ]
        )

        #Train model
        pipeline.fit(X_train, y_train)

        #Predict
        y_pred = pipeline.predict(X_test)

        #Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision_macro = precision_score(y_test, y_pred, average="macro")
        recall_macro = recall_score(y_test, y_pred, average="macro")
        f1_macro = f1_score(y_test, y_pred, average="macro")
       
        vocab_size = len(pipeline.named_steps["tfidf"].vocabulary_)

        #log parameters
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("vectorizer", "TF-IDF")
        mlflow.log_param("stop_words", "english")

        #log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision_macro", precision_macro)
        mlflow.log_metric("recall_macro", recall_macro)
        mlflow.log_metric("f1_macro", f1_macro)  
        mlflow.log_metric("vocabulary_size", vocab_size)

   
        class SupportTicketModel(mlflow.pyfunc.PythonModel):

            def load_context(self, context):
                self.pipeline = joblib.load(context.artifacts["pipeline"])

            def predict(self, context, model_input):
                return self.pipeline.predict(model_input["cleaned_ticket_text"])

     
        # Save trained pipeline as pickle/joblib file
        with tempfile.TemporaryDirectory() as tmp_dir:

            pipeline_path = f"{tmp_dir}/pipeline.joblib"
            joblib.dump(pipeline, pipeline_path)

            # Create input example as DataFrame
            input_example = X_train.head(5).to_frame(name="cleaned_ticket_text")

            # Get prediction example using DataFrame
            prediction_example = pipeline.predict(input_example["cleaned_ticket_text"])

            # Infer signature
            signature = infer_signature(
                input_example,
                prediction_example
            )

            # Log wrapped pyfunc model
            mlflow.pyfunc.log_model(
                artifact_path="support_ticket_model",
                python_model=SupportTicketModel(),
                artifacts={
                    "pipeline": pipeline_path
                },
                signature=signature,
                input_example=input_example
            )
        
        #Capture the run_id
        run_id = mlflow.active_run().info.run_id        

        #append results to the delta table. Delta Table = Business Summary
        results.append({
            "run_id": run_id,
            "model": model_name,
            "vectorizer": "TF-IDF",
            "vocabulary_size": vocab_size,
            "accuracy": accuracy,
            "precision_macro": precision_macro,
            "recall_macro": recall_macro,
            "f1_macro": f1_macro
        })

results_df = pd.DataFrame(results)

results_spark = spark.createDataFrame(results_df)

results_spark.write.mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.model_training_results")

###### Notebook Summary

- Created bronze, silver and gold tables.
- Converted gold tables to pandas.
- Train/test split to  generated model results for the below classifiers
     - Logistic Regression 
     - Decision tree 
     - Random Forest classifiers
- Dynamically retrieved the Databricks user and created a personal MLflow experiment.
- Logged the following to MLflow:
     - Parameters
     - Metrics
     - Input Example
     - Model Signature
     - Model Artifact

- Appended results to the delta table - dbw_agentic_ai_dev.support_ticket_ai.model_training_results

######  Key Learnings

- Importance of MLflow Experiments
- Difference between MLflow Run and Experiment
- Purpose of artifact_path
- Purpose of infer_signature()
- Importance of input_example
- Difference between Parameters and Metrics
- Why model artifacts contain MLmodel, model.pkl and environment files
- Why macro averaging is used for multi-class classification


###### Real-world Applications

The techniques demonstrated in this notebook are commonly used for:

- IT Support Ticket Classification
- Email Routing
- Customer Complaint Categorization
- Help Desk Automation
- Document Classification

###### Next Notebook

18_nlp_model_evaluation

Purpose

Compare all trained models,identify the best model and save the best model metadata for registration.